# 06 — Biological Validation

GO enrichment and STRING network analysis of ML-selected proteins.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.config import *
from src.data_extraction import load_all_data
from src.models import ENDOSOMAL_PROTEINS
from src.biological_validation import (run_go_enrichment, get_string_network,
    compare_enrichment, check_endosomal_enrichment, _placeholder_enrichment)
from src.figures import set_publication_style, plot_go_enrichment
set_publication_style()

In [2]:
data = load_all_data()
sig = data['sig_proteins']
selected_genes = sig['mouse_gene'].tolist()[:20]  # Use top 20 for demo
background = data['protein_list']['mouse_gene'].tolist()
print(f'Selected: {len(selected_genes)}, Background: {len(background)}')

[data_extraction] Loading real data from aba6334_data_file_s1.xlsx and aba6334_data_file_s3.xlsx


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/d

  Proteins:      1505
  Significant:   71
  Mouse samples: 11
  Human subjects:58
  Data source:   supplement
Selected: 20, Background: 1505


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix[gene] = vals
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix[gene] = vals
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result o

In [3]:
# GO enrichment (using placeholder if API unavailable)
print('Running GO enrichment for selected proteins...')
try:
    enrich_sel = run_go_enrichment(selected_genes, organism='mmusculus')
except Exception as e:
    print(f'API failed: {e}; using placeholder')
    enrich_sel = _placeholder_enrichment(selected_genes)
enrich_sel.to_csv(RESULTS_DIR / 'go_enrichment_results.csv', index=False)
print(f'Terms found: {len(enrich_sel)}')
enrich_sel.head()

Running GO enrichment for selected proteins...
Terms found: 82


,description,effective_domain_size,goshv,intersection_size,intersections,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size,source_order,group_id,term_id
0,"""The process in which the anatomical structure...",25907,23045,9,"[[IBA], [IMP], [IMP, IGI, ISS], [IMP, IGI, IBA...",neuron projection morphogenesis,GO:0048812,0.000001,"[GO:0031175, GO:0120039]",0.45,query_1,20,0.012748,True,GO:BP,706,12891,11,GO:0048812
1,"""The process in which the anatomical structure...",25907,29365,9,"[[IBA], [IMP], [IMP, IGI, ISS], [IMP, IGI, IBA...",plasma membrane bounded cell projection morpho...,GO:0120039,0.000002,[GO:0048858],0.45,query_1,20,0.012483,True,GO:BP,721,19211,11,GO:0120039
2,"""The process in which the anatomical structure...",25907,23087,9,"[[IBA], [IMP], [IMP, IGI, ISS], [IMP, IGI, IBA...",cell projection morphogenesis,GO:0048858,0.000002,"[GO:0000902, GO:0009653, GO:0030030]",0.45,query_1,20,0.012363,True,GO:BP,728,12933,11,GO:0048858
3,"""De novo generation of a long process of a neu...",25907,13045,8,"[[IBA], [IMP], [IMP, IGI, ISS], [IMP, IGI, IBA...",axonogenesis,GO:0007409,0.000002,"[GO:0048667, GO:0048812, GO:0061564]",0.40,query_1,20,0.016598,True,GO:BP,482,2891,11,GO:0007409
4,"""The developmental process in which the size o...",25907,10392,10,"[[IBA], [IMP], [IMP, IGI, ISS], [IMP, IGI, IBA...",cell morphogenesis,GO:0000902,0.000002,[GO:0009653],0.50,query_1,20,0.009497,True,GO:BP,1053,238,11,GO:0000902


In [4]:
# Check endosomal GO term enrichment
endo_terms = check_endosomal_enrichment(enrich_sel)
if len(endo_terms):
    print('Endosomal GO terms enriched:')
    print(endo_terms[['term_id', 'term_name', 'p_value']].to_string(index=False))
else:
    print('No endosomal GO terms enriched (expected in placeholder mode)')

No endosomal GO terms enriched (expected in placeholder mode)


In [5]:
# GO dot plot
plot_go_enrichment(enrich_sel)
print('GO enrichment plot saved')

/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/figures.py:73: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/figures.py:73: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")


[figures] Saved fig5_go_enrichment
GO enrichment plot saved


In [6]:
# STRING network
print('\nFetching STRING network...')
try:
    network = get_string_network(selected_genes[:10], species=10090)
    if len(network):
        print(f'Interactions: {len(network)}')
        network.to_csv(RESULTS_DIR / 'string_network.csv', index=False)
        print(network.head())
    else:
        print('No interactions returned (check network connectivity)')
except Exception as e:
    print(f'STRING failed: {e}')


Fetching STRING network...
Interactions: 21
                 stringId_A                stringId_B preferredName_A  \
0  10090.ENSMUSP00000005406  10090.ENSMUSP00000063933             App   
1  10090.ENSMUSP00000005406  10090.ENSMUSP00000121203             App   
2  10090.ENSMUSP00000005406  10090.ENSMUSP00000006828             App   
3  10090.ENSMUSP00000005406  10090.ENSMUSP00000006235             App   
4  10090.ENSMUSP00000005406  10090.ENSMUSP00000072428             App   

  preferredName_B  ncbiTaxonId  score  nscore  fscore  pscore  ascore  escore  \
0            Chl1        10090  0.486       0       0   0.000   0.109   0.281   
1            Ctsd        10090  0.683       0       0   0.000   0.047   0.144   
2           Aplp1        10090  0.709       0       0   0.070   0.128   0.628   
3            Ctsb        10090  0.791       0       0   0.000   0.104   0.000   
4           Aplp2        10090  0.851       0       0   0.064   0.298   0.631   

   dscore  tscore  
0     0.0